# Imports

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"

LANDING_SCHEMA = "raw"
LANDING_VOLUME = "landing"

OPS_SCHEMA = "ops"
STATE_VOLUME = "pipeline_state"


LANDING_ROOT = (
    f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{LANDING_VOLUME}"
)

STATE_ROOT = (
    f"/Volumes/{CATALOG}/{OPS_SCHEMA}/{STATE_VOLUME}/bronze"
)

BRONZE_DATABASE = f"{CATALOG}.{BRONZE_SCHEMA}"

print(f"Landing: {LANDING_ROOT}")
print(f"State:   {STATE_ROOT}")
print(f"Bronze:  {BRONZE_DATABASE}")

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{OPS_SCHEMA}
""")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOG}.{OPS_SCHEMA}.{STATE_VOLUME}
""")

In [0]:
def add_bronze_metadata(df, source_entity: str):

    return (
        df.select(
            "*",

            col("_metadata.file_path")
                .alias("_source_file"),

            col("_metadata.file_name")
                .alias("_source_file_name"),

            col("_metadata.file_size")
                .alias("_source_file_size"),

            col("_metadata.file_modification_time")
                .alias("_source_file_modification_time")
        )
        .withColumn(
            "_source_system",
            lit("FastF1")
        )
        .withColumn(
            "_source_entity",
            lit(source_entity)
        )
        .withColumn(
            "_ingested_at",
            current_timestamp()
        )
    )

In [0]:
def ingest_bronze(
    entity: str,
    partition_columns: list[str]
):
    """
    Ingesta incremental desde Landing hacia Bronze.

    Source:
        Unity Catalog Volume + Parquet

    Target:
        Unity Catalog Managed Delta Table

    Auto Loader mantiene el estado de archivos procesados
    mediante checkpoint.
    """

    source_path = f"{LANDING_ROOT}/{entity}"

    target_table = f"{BRONZE_DATABASE}.{entity}"

    schema_location = (
        f"{STATE_ROOT}/{entity}/schema"
    )

    checkpoint_location = (
        f"{STATE_ROOT}/{entity}/checkpoint"
    )

    print(f"Source: {source_path}")
    print(f"Target: {target_table}")

    reader = (
        spark.readStream
        .format("cloudFiles")
        .option(
            "cloudFiles.format",
            "parquet"
        )
        .option(
            "cloudFiles.schemaLocation",
            schema_location
        )
        .option(
            "cloudFiles.includeExistingFiles",
            "true"
        )
    )

    # display(reader)

    # Indicamos explícitamente las columnas Hive:
    #
    # season=2025
    # round=01

    if partition_columns:
        reader = reader.option(
            "cloudFiles.partitionColumns",
            ",".join(partition_columns)
        )

    df = reader.load(source_path)

    df = add_bronze_metadata(
        df,
        source_entity=entity
    )

    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            checkpoint_location
        )
        .option(
            "mergeSchema",
            "true"
        )
        .trigger(
            availableNow=True
        )
        .toTable(target_table)
    )

    query.awaitTermination()

    print(f"Completed: {target_table}")

# Events

Aquí se insertan los calendarios de cada año

In [0]:
# COMMAND ----------

BRONZE_SOURCES = {
    "events": [
        "season"
    ],

    "laps": [
        "season",
        "round"
    ],

    "qualifying_results": [
        "season",
        "round"
    ],

    "race_results": [
        "season",
        "round"
    ],

    "weather": [
        "season",
        "round"
    ]
}

In [0]:
for entity, partition_columns in BRONZE_SOURCES.items():

    print("=" * 80)
    print(f"Processing {entity}")
    print("=" * 80)

    ingest_bronze(
        entity=entity,
        partition_columns=partition_columns
    )

# Validaciones de tablas

In [0]:
for entity in BRONZE_SOURCES:

    table_name = f"{BRONZE_DATABASE}.{entity}"

    print(f"\n===== {table_name} =====")

    spark.sql(f"""
        SELECT COUNT(*) AS rows
        FROM {table_name}
    """).show()